In [16]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [17]:
block_size = 512
batch_size = 16
learning_rate = 0.001
weight_decay = 1e-3
dropout = 0.3  # THE KEY CHANGE
vocab_size = 10
n_head = 4
n_layer = 4
n_embd = 128

def discretize(y_norm, precision=3, base=10):
    tokens = []
    for val in y_norm:
        val = np.clip(val, 0, 1 - 1e-9)
        digits = []
        remaining = val
        for _ in range(precision):
            remaining *= base
            digit = int(remaining)
            digits.append(digit)
            remaining -= digit
        tokens.extend(digits)
    return tokens

def undiscretize(tokens, precision=3, base=10):
    values = []
    for i in range(0, len(tokens), precision):
        chunk = tokens[i:i+precision]
        if len(chunk) < precision:
            break
        val = 0
        for j, d in enumerate(chunk):
            val += d / (base ** (j + 1))
        values.append(val)
    return np.array(values)

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout_layer = nn.Dropout(dropout)
        self.head_size = head_size
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * self.head_size ** -0.5  # FIXED scaling
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout_layer(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout_layer = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout_layer(self.proj(out))

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.GELU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class SignalTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

print(f"Model defined with dropout={dropout}")

Model defined with dropout=0.3


In [18]:
A = 1.0
omega = 2 * np.pi

def get_batch(sequences, split='train'):
    n = int(0.9 * len(sequences))
    data = sequences[:n] if split == 'train' else sequences[n:]
    while True:
        seq_indices = torch.randint(len(data), (batch_size,))
        x_batch, y_batch = [], []
        for idx in seq_indices:
            sequence = data[idx]
            if len(sequence) >= block_size + 1:
                start_idx = torch.randint(0, len(sequence) - block_size, (1,))
                x_batch.append(sequence[start_idx:start_idx + block_size])
                y_batch.append(sequence[start_idx + 1:start_idx + block_size + 1])
        if len(x_batch) > 0:
            return torch.stack(x_batch).to(device), torch.stack(y_batch).to(device)

@torch.no_grad()
def estimate_loss(model, sequences):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(200)
        for k in range(200):
            X, Y = get_batch(sequences, split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

def generate_training_sinusoids(num_signals, signal_length, A, omega, noise_std=0.0):
    all_tokens = []
    for i in range(num_signals):
        phase = np.random.uniform(0, 2 * np.pi)
        t = np.arange(signal_length) * 0.01
        y = A * np.sin(omega * t + phase)
        if noise_std > 0:
            y += np.random.normal(0, noise_std, signal_length)
        y_range = A + 3 * noise_std if noise_std > 0 else A
        y_norm = (y - (-y_range)) / (2 * y_range)
        y_norm = np.clip(y_norm, 0, 1 - 1e-9)
        tokens = discretize(y_norm)
        all_tokens.append(torch.tensor(tokens, dtype=torch.long))
    return all_tokens, y_range

print("Helper functions defined.")

Helper functions defined.


In [19]:
for sigma in [0.0, 0.5]:
    print(f"\n{'='*50}")
    print(f"DROPOUT=0.3 — sinusoid σ={sigma}")
    print(f"{'='*50}")
    
    np.random.seed(42)
    torch.manual_seed(42)
    signals, y_range = generate_training_sinusoids(500, 256, A, omega, noise_std=sigma)
    
    model = SignalTransformer().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    
    for iter in range(20000):
        xb, yb = get_batch(signals, 'train')
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        
        if iter % 5000 == 0 or iter == 19999:
            losses = estimate_loss(model, signals)
            print(f"  iter {iter}: train {losses['train']:.4f}, val {losses['val']:.4f}")
    
    model.eval()
    t_ctx = np.arange(50) * 0.01
    y_ctx = A * np.sin(omega * t_ctx)
    y_ctx_norm = (y_ctx - (-y_range)) / (2 * y_range)
    y_ctx_norm = np.clip(y_ctx_norm, 0, 1 - 1e-9)
    ctx_tokens = discretize(y_ctx_norm)
    ctx_tensor = torch.tensor(ctx_tokens, dtype=torch.long).unsqueeze(0).to(device)
    
    with torch.no_grad():
        gen = model.generate(ctx_tensor, 600)
    
    gen_tokens = gen.tolist()[0]
    gen_values_norm = undiscretize(gen_tokens)
    gen_values = gen_values_norm * (2 * y_range) + (-y_range)
    generated = gen_values[50:]
    
    y_true = A * np.sin(omega * np.arange(50, 50 + len(generated)) * 0.01)
    mae = np.mean(np.abs(y_true[:len(generated)] - generated[:len(y_true)]))
    
    print(f"\n  MAE: {mae:.4f}")
    print(f"  Compare:")
    print(f"    No dropout (old):   σ={sigma} MAE ≈ {'0.014' if sigma == 0 else '0.526'}")
    print(f"    Dropout=0.3:        σ={sigma} MAE = {mae:.4f}")
    print(f"    Val loss: {losses['val'].item():.4f}")
    print(f"    Val loss exploded: {'YES' if losses['val'].item() > 2.0 else 'NO'}")


DROPOUT=0.3 — sinusoid σ=0.0
  iter 0: train 2.3579, val 2.3585
  iter 5000: train 0.1060, val 0.1219
  iter 10000: train 0.0671, val 0.0885
  iter 15000: train 0.0531, val 0.0755
  iter 19999: train 0.0345, val 0.0434

  MAE: 0.0759
  Compare:
    No dropout (old):   σ=0.0 MAE ≈ 0.014
    Dropout=0.3:        σ=0.0 MAE = 0.0759
    Val loss: 0.0434
    Val loss exploded: NO

DROPOUT=0.3 — sinusoid σ=0.5
  iter 0: train 2.3794, val 2.3771
  iter 5000: train 2.0174, val 2.0229
  iter 10000: train 1.9925, val 2.0378
  iter 15000: train 1.8729, val 2.1114
  iter 19999: train 1.7638, val 2.1632

  MAE: 0.4338
  Compare:
    No dropout (old):   σ=0.5 MAE ≈ 0.526
    Dropout=0.3:        σ=0.5 MAE = 0.4338
    Val loss: 2.1632
    Val loss exploded: YES
